# [LAB] RAG 실습

보험 약관 PDF 한 권으로 RAG를 처음부터 끝까지 만들어 봅니다. 장마다 RAG 핵심 모듈을 하나씩 더하고, 표시된 값을 바꿔 결과가 어떻게 달라지는지 봅니다.

- **2장 PDF 불러오기와 청크 나누기**: Loading과 Splitting입니다. PDF를 읽고 청크로 나눕니다.
- **3장 임베딩과 유사도**: Embedding입니다. 문장을 벡터로 바꾸고 뜻이 가까운 정도를 숫자로 비교합니다.
- **4장 벡터 저장소와 검색**: Storage와 Retrieval입니다. 청크를 Chroma에 넣고 질문과 가까운 청크를 찾습니다.
- **5장 RAG 체인**: Generation입니다. 검색 결과를 근거로 답변하는 체인을 완성하고 스트리밍으로 받습니다.
- **6장 RAGaaS**: 청킹·임베딩·저장을 API가 대신 하는 방식으로 같은 일을 합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**목차**

- 1. 실습 방법
- 2. PDF 불러오기와 청크 나누기
  - 2.1 PDF를 Document로 불러오기
  - 2.2 Document를 청크로 나누기
  - 2.3 청크 확인하기
- 3. 임베딩과 유사도
  - 3.1 사내 임베딩을 LangChain에 맞추기
  - 3.2 문장을 벡터로 바꾸기
  - 3.3 유사도 비교와 질문 검색
- 4. 벡터 저장소와 검색
  - 4.1 벡터 저장소 만들기
  - 4.2 retriever로 바꾸기
  - 4.3 질문으로 청크 찾기
- 5. RAG 체인
  - 5.1 벡터 저장소 만들기
  - 5.2 retriever로 바꾸기
  - 5.3 모델과 프롬프트 준비
  - 5.4 검색 결과를 context로 합치기
  - 5.5 RAG 체인 연결하기
  - 5.6 질문하고 스트리밍으로 답변 받기
- 6. RAGaaS
  - 6.1 1부: 업로드할 문서 준비
  - 6.2 2부: 문서 검색
  - 6.3 3부: 검색 결과로 답변 만들기
  - 6.4 4부: 문서 삭제(실행하지 않음)
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER", "RAG_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. PDF 불러오기와 청크 나누기

약관 PDF를 읽어 `Document`로 만들고, 검색하기 좋은 크기의 청크로 나눕니다. RAG 핵심 모듈 중 Loading과 Splitting입니다. 이 장은 API를 호출하지 않습니다.

`# ✍️ CHANGE_HERE:` 줄의 청크 크기를 바꿔 결과가 어떻게 달라지는지 봅니다.

**왜 배우나:** 청크 크기는 검색 정확도와 문맥 보존 사이의 균형을 정합니다. 4·5장은 준비 코드에서 이 장의 원래 설정(`chunk_size=500`, `chunk_overlap=75`)으로 PDF를 다시 나눠 씁니다. 여기서 바꾼 값은 뒤 장에 이어지지 않습니다.

준비 코드입니다. PDF를 읽는 도구와 나누는 도구를 불러오고, 약관 PDF 파일 경로(`PDF_PATH`)를 정합니다. 펼쳐 보지 않고 실행만 해도 됩니다.

In [ ]:
"""
데이터 로딩과 청킹: PDF를 Document와 Chunk로 변환하기

핵심 개념:
  - Document = LangChain이 문서 본문과 메타데이터를 다루는 기본 단위
  - Chunking = 긴 문서를 검색 가능한 작은 조각으로 나누는 과정
  - chunk_size와 chunk_overlap은 검색 정확도와 문맥 보존 사이의 균형을 조정

"""

from pathlib import Path

import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


PDF_PATH = Path("data/meritz_pet_insurance.pdf")

### 2.1 PDF를 Document로 불러오기

PDF 파일을 읽어 `Document` 목록을 만듭니다. `Document` 하나에는 본문(`page_content`)과 메타데이터(`metadata`)가 들어 있습니다. 파일 경로(`PDF_PATH`)와 읽는 도구(`PDFMinerLoader`)는 준비 코드에 있습니다.

In [ ]:
# ── PDF 로딩 ─────────────────────────────────────────────────────────────────
# PDFMinerLoader: PDF 파일을 LangChain Document 목록으로 변환
# 각 Document는 page_content와 metadata를 가진다.
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()

### 2.2 Document를 청크로 나누기

**✍️ CHANGE_HERE: chunk_size를 500에서 200으로 줄이면 청크 수와 청크 길이는 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `500` 자리에 복사해 붙여 넣기

```text
200
```

이 셀과 2.3 셀을 다시 실행합니다. 확인했으면 `500`으로 되돌립니다.

바로 위에서 만든 `text_splitter`로 `documents`를 청크로 나눕니다. `chunk_size`는 청크 하나의 최대 길이, `chunk_overlap`은 이웃한 청크끼리 겹쳐 두는 길이입니다.

In [ ]:
# ── 문서 청킹 ─────────────────────────────────────────────────────────────────
# 전체 문서는 한 번에 LLM에 넣기 어렵기 때문에 검색 가능한 Chunk로 나눈다.
# RecursiveCharacterTextSplitter: 문단 → 줄 → 단어 → 문자 순서로 분할 지점을 찾는다.
# chunk_size: 각 Chunk의 최대 길이
# chunk_overlap: 인접 Chunk 사이에 겹쳐 둘 길이
text_splitter = RecursiveCharacterTextSplitter(
    # 작을수록 검색 단위가 정밀하지만 전체 문맥이 줄어든다.
    # ✍️ CHANGE_HERE: chunk_size를 500에서 200으로 줄이면 청크 수와 청크 길이는 어떻게 될까요?
    chunk_size=500,
    # 클수록 문맥 연결성이 좋아지지만 중복과 비용이 늘어난다.
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

<details><summary>무슨 일이 생기나</summary>

- 바꾸기 전(`500`): `청크 수: 50`, 앞 세 청크의 `len=`은 493, 487, 490입니다.
- 바꾼 뒤(`200`): `청크 수: 160`으로 세 배 넘게 늘고, `len=`은 181, 176, 160으로 줄어듭니다. 나누는 자리가 달라져 `1.`·`2.` 청크의 내용도 바뀝니다.
- 청크가 작으면 한 조각이 짧아 앞뒤 문맥이 잘립니다. 대신 조각 수가 늘어 임베딩할 양도 늘어납니다.

</details>

### 2.3 청크 확인하기

In [ ]:
# ── 결과 확인 ─────────────────────────────────────────────────────────────────
print(f"청크 수: {len(chunks)}")
print()
for index, chunk in enumerate(chunks[:3]):
    text = " ".join(chunk.page_content.split())
    print(f"{index}. len={len(chunk.page_content)} \n- {text[:120]}...")
    print()

**볼 지점**
- `청크 수`는 PDF가 몇 조각으로 나뉘었는지입니다.
- `len=` 뒤의 숫자는 청크의 글자 수입니다. `chunk_size`보다 크지 않습니다.
- 청크 내용은 앞 120자만 보여 줍니다.
- 청크가 작으면 한 조각에 담긴 내용이 줄고, 조각 수가 늘어 임베딩할 양도 늘어납니다.

## 3. 임베딩과 유사도

문장을 임베딩(의미를 숫자 목록, 곧 벡터로 바꾸는 일)하고, 벡터끼리 얼마나 가까운지 코사인 유사도로 비교합니다. RAG에서 "관련 문서 검색"은 이 벡터 거리를 계산하는 일입니다.

코드는 완성되어 있습니다. 차례로 실행하고, 마지막 셀의 `# ✍️ CHANGE_HERE:` 줄에서 질문을 바꿔 봅니다.

첫 셀의 `SamsungEmbeddings`는 사내 임베딩 API를 LangChain에서 쓸 수 있게 맞춘 클래스입니다. LangChain의 `Embeddings`를 물려받아 `embed_documents`(여러 문장)와 `embed_query`(질문 하나) 두 메서드만 만들면, 벡터 저장소나 검색기 같은 LangChain 도구에서 그대로 쓸 수 있습니다. 셀 아래쪽의 `if`는 연결 환경에 맞는 임베딩을 고르는 코드라 건너뛰어도 됩니다.

준비 코드입니다. 필요한 도구를 불러오고 설정 파일을 읽습니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
"""
임베딩(Embedding): 텍스트를 숫자 벡터로 변환하기

핵심 개념:
  - 임베딩 = 텍스트의 '의미'를 숫자 벡터로 표현
  - 의미가 비슷한 텍스트는 벡터 공간에서 가까이 위치
  - RAG에서 '관련 문서 검색'은 이 벡터 거리를 계산하는 것

"""

import os

import numpy as np
from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다
from config.embeddings import get_embeddings
from config.samsung import embedding_config

load_dotenv()

import requests
from langchain_core.embeddings import Embeddings

### 3.1 사내 임베딩을 LangChain에 맞추기

In [ ]:
BATCH = 10  # API 한 번에 처리 가능한 최대 텍스트 수


class SamsungEmbeddings(Embeddings):
    def embed_documents(self, texts):
        # BATCH 개씩 나눠서 API 호출, 결과를 하나로 합침
        result = []
        for i in range(0, len(texts), BATCH):
            config = embedding_config()
            res = requests.post(
                config["url"],
                json={"input": texts[i : i + BATCH]},
                headers=config["headers"],
                timeout=30,
            )
            res.raise_for_status()
            result.extend(item["embedding"] for item in res.json()["data"])
        return result

    def embed_query(self, text):
        # 질문 하나를 임베딩할 때는 embed_documents 재활용
        return self.embed_documents([text])[0]


provider = os.getenv("EMBED_PROVIDER", "samsung").strip().lower()
if provider == "samsung":
    embedding_config()  # 설정이 비어 있으면 여기서 멈춘다.
    embeddings = SamsungEmbeddings()
else:
    # 사외에서는 같은 유사도 실습을 OpenAI 임베딩으로 실행한다.
    embeddings = get_embeddings()

### 3.2 문장을 벡터로 바꾸기

In [ ]:
sentences = [
    "강아지가 공원에서 뛰어놀고 있다.",
    "개가 잔디밭에서 달리고 있다.",
    "오늘 주식시장이 크게 하락했다.",
]

# embed_documents(): 텍스트 리스트 → 벡터 리스트
vectors = embeddings.embed_documents(sentences)

print("=== 벡터 기본 정보 ===")
print(f"문장 수    : {len(vectors)}")
print(f"벡터 차원  : {len(vectors[0])}")
print(f"앞 5개 값  : {[v[:5] for v in vectors]}")
print()

### 3.3 유사도 비교와 질문 검색

**✍️ CHANGE_HERE: 질문을 같은 낱말이 없는 문장이나 다른 언어로 바꾸면 가장 유사한 문장은 무엇이 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `애완동물이 뛰노는 모습` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

한국어(세 문장과 같은 낱말이 없음)
```text
증시가 폭락했다
```

영어
```text
Stock prices plunged today
```

일본어
```text
株価が暴落した
```

중국어
```text
股市暴跌
```

이 셀을 다시 실행합니다. 확인했으면 `애완동물이 뛰노는 모습`으로 되돌립니다.

In [ ]:
# ── 코사인 유사도 ─────────────────────────────────────────────────────────────
# 코사인 유사도: 1에 가까울수록 의미가 유사, 0에 가까울수록 관련 없음
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


v = [np.array(vec) for vec in vectors]
print("=== 문장 간 유사도 ===")
print(f"[강아지] vs [개]   : {cosine_similarity(v[0], v[1]):.4f}")  # 높아야 함
print(f"[강아지] vs [주식] : {cosine_similarity(v[0], v[2]):.4f}")  # 낮아야 함
print()

# embed_query(): 질문 하나를 벡터로 변환 (검색 시 사용)
# ✍️ CHANGE_HERE: 질문을 같은 낱말이 없는 문장이나 다른 언어로 바꾸면 가장 유사한 문장은 무엇이 될까요?
query = "애완동물이 뛰노는 모습"
query_vec = np.array(embeddings.embed_query(query))
scores = [cosine_similarity(query_vec, v_i) for v_i in v]
best = scores.index(max(scores))
print(f'질문: "{query}"')
print(f'가장 유사한 문장: "{sentences[best]}" (유사도: {scores[best]:.4f})')
print()

<details><summary>무슨 일이 생기나</summary>

- 바꾸기 전("애완동물이 뛰노는 모습"): 강아지 문장이 가장 유사합니다(유사도 0.2614).
- "증시가 폭락했다": "오늘 주식시장이 크게 하락했다."가 가장 유사합니다(0.2877). 증시·폭락과 주식시장·하락은 낱말은 다르지만 뜻이 같습니다.
- 영어(0.4580), 일본어(0.3993), 중국어(0.4105): 셋 다 같은 주식 문장이 가장 유사합니다.
- 임베딩은 글자가 아니라 뜻을 숫자로 바꿉니다. 그래서 같은 낱말이 없어도, 언어가 달라도 뜻이 가까운 문장을 찾습니다. 유사도 숫자는 임베딩 모델마다 다릅니다.

</details>

**볼 지점**
- `벡터 차원`은 문장 하나를 바꾼 숫자 목록의 길이입니다.
- 유사도는 1에 가까울수록 뜻이 비슷하고, 0에 가까울수록 관련이 없습니다.
- 유사도 숫자는 임베딩 모델마다 다릅니다. 숫자 자체보다 어느 쪽이 더 큰지를 봅니다.
- 질문 "애완동물이 뛰노는 모습"에는 "강아지"나 "개"라는 낱말이 없습니다. 그래도 뜻이 가까운 문장을 찾는지 봅니다.

## 4. 벡터 저장소와 검색

약관 PDF를 2장과 같은 설정으로 나눈 청크를 임베딩해 Chroma 벡터 저장소에 넣고, 질문과 가까운 청크 3개를 찾아봅니다. RAG 핵심 모듈 중 Embedding, Storage, Retrieval입니다. PDF를 읽고 나누는 코드는 준비 코드에 들어 있습니다.

4.2에서는 가져올 청크 수(`k`)를, 4.3에서는 질문을 `# ✍️ CHANGE_HERE:` 줄에서 바꿔 검색 결과가 어떻게 달라지는지 봅니다.

**왜 배우나:** 검색 결과가 틀리면 생성 단계가 좋아도 답변이 흔들립니다. 답변을 만들기 전에 검색 결과만 따로 확인해 봅니다.

준비 코드입니다. 필요한 도구를 불러오고, 2장과 같은 방법으로 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

### 4.1 벡터 저장소 만들기

청크를 벡터로 바꿔 Chroma 벡터 저장소에 넣습니다. 임베딩 모델은 준비 코드에서 불러온 `get_embeddings()`로 가져옵니다. 셀 첫 줄의 `Chroma().delete_collection()`은 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게 저장소를 비우는 줄이라 지우지 않습니다.

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)

### 4.2 retriever로 바꾸기

**✍️ CHANGE_HERE: 가져올 청크 수 k를 3에서 1이나 5로 바꾸면 4.3의 검색 결과는 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `"k": 3` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

```text
"k": 1
```

```text
"k": 5
```

이 셀과 4.3 셀을 다시 실행합니다. 확인했으면 `"k": 3`으로 되돌리고 이 셀을 다시 실행합니다.

벡터 저장소를 검색기(retriever)로 바꿉니다. retriever는 질문을 받으면 가까운 청크를 찾아 돌려주는 객체입니다. 한 번에 가져올 청크 수 `k`는 검색 설정 `search_kwargs` 안에 넣습니다. `as_retriever(k=3)`처럼 바깥에 쓰면 오류 없이 무시되고 기본값 4개를 가져옵니다.

In [ ]:
# ✍️ CHANGE_HERE: 가져올 청크 수 k를 3에서 1이나 5로 바꾸면 4.3의 검색 결과는 어떻게 될까요?
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

<details><summary>무슨 일이 생기나</summary>

- `1`: 4.3에 `0.` 청크 하나만 나옵니다.
- `5`: `0.`~`4.` 다섯 개가 나옵니다. `0.`~`2.`는 `3`일 때와 같고, 덜 가까운 청크 두 개가 뒤에 붙습니다.
- `k`가 작으면 답에 필요한 내용이 빠질 수 있고, 크면 관계가 먼 청크까지 섞여 프롬프트가 길어집니다.

</details>

### 4.3 질문으로 청크 찾기

**✍️ CHANGE_HERE: 질문을 다른 언어나 약관에 없는 내용으로 바꾸면 어떤 청크가 나올까요?**

**바꿔 볼 값** — 바로 아래 줄의 `펫보험 보장 내용이 뭐야?` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

한국어
```text
청약을 철회할 수 있는 기간은 며칠이야?
```

영어(위와 같은 질문)
```text
How many days do I have to withdraw my insurance application?
```

일본어(위와 같은 질문)
```text
保険の申込みを撤回できる期間は何日ですか？
```

중국어(위와 같은 질문)
```text
撤回保险申请的期限是几天？
```

약관에 없는 질문
```text
강아지에게 초콜릿을 먹여도 돼?
```

이 셀만 다시 실행하고, 네 언어의 질문은 `0.` 청크를 서로 비교합니다. 확인했으면 원래 질문으로 되돌립니다.

In [ ]:
print("retriever ready")
# ✍️ CHANGE_HERE: 질문을 다른 언어나 약관에 없는 내용으로 바꾸면 어떤 청크가 나올까요?
question = "펫보험 보장 내용이 뭐야?"
results = retriever.invoke(question)
for index, doc in enumerate(results):
    text = " ".join(doc.page_content.split())
    print(f"{index}. len={len(doc.page_content)} \n- {text[:180]}...")
    print()

<details><summary>무슨 일이 생기나</summary>

- 바꾸기 전("펫보험 보장 내용이 뭐야?"): 상품 요약과 보장·유의사항 청크 3개가 나옵니다.
- 한국어: `0.`에 `청약철회 기간 청약일 보험증권을 받은 날 … +15일`로 시작하는 청크가 나옵니다.
- 영어·일본어·중국어: 세 언어 모두 `0.`에 한국어 질문과 같은 청약철회 청크가 나옵니다(`1.`·`2.`는 언어마다 조금 다릅니다). 질문과 약관이 같은 낱말을 하나도 쓰지 않아도 뜻이 가까우면 찾습니다. 임베딩 모델에 따라 다른 언어 질문의 결과는 달라질 수 있습니다.
- 약관에 없는 질문: 그래도 청크 3개가 나옵니다(보험규정 안내, 계약 부활, 반려견 의료비 표처럼 초콜릿과 관계없는 내용). 검색기는 "없다"고 하지 않고 늘 가장 가까운 `k`개를 돌려줍니다. 5장에서 이 청크를 받은 모델이 어떻게 답하는지 봅니다.

</details>

**볼 지점**
- 결과는 질문과 가까운 순서로 나옵니다. `0.`이 가장 가깝습니다.
- `len=`은 청크의 글자 수이고, 내용은 앞 180자만 보여 줍니다.
- 질문과 낱말이 똑같지 않아도 뜻이 가까운 청크가 나오는지 봅니다.

## 5. RAG 체인

4장까지 만든 검색에 LLM을 이어, 약관을 근거로 답변하는 RAG 체인을 완성합니다. 검색 결과를 `context` 문자열로 합치고, 검색 → 프롬프트 → 모델 → 출력 파서를 `|`로 연결합니다. 답변은 스트리밍(조각이 생기는 대로 받는 방식)으로 출력합니다.

벡터 저장소와 retriever는 4장과 같은 코드입니다. 마지막 셀에서는 `# ✍️ CHANGE_HERE:` 줄의 질문을, 프롬프트 셀에서는 system 지시문을 바꿔 봅니다.

**왜 배우나:** RAG의 Generation 단계입니다. "검색된 근거 안에서만 답하라"는 지시가 프롬프트에 어떻게 들어가고, 답변을 어떻게 바꾸는지 직접 봅니다.

준비 코드입니다. 필요한 도구를 불러오고, 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 4장 준비 코드와 같은 내용이라 실행만 하면 됩니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

### 5.1 벡터 저장소 만들기

4장에서 한 일을 다시 합니다. 청크를 임베딩해 Chroma 벡터 저장소를 만듭니다. 셀 첫 줄의 `Chroma().delete_collection()`은 저장소를 비우는 줄이라 지우지 않습니다.

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)

### 5.2 retriever로 바꾸기

벡터 저장소를 질문과 가까운 청크 3개를 돌려주는 retriever로 바꿉니다.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### 5.3 모델과 프롬프트 준비

**✍️ CHANGE_HERE: 지시문 끝에 '문서에서 확인할 수 없습니다' 규칙을 더하면 약관에 없는 질문의 답은 어떻게 바뀔까요?**

**바꿔 볼 값** — 바로 아래 줄의 `답하세요.` 자리에 복사해 붙여 넣기

```text
답하세요. 맥락에 답이 없으면 '문서에서 확인할 수 없습니다'라고 말하세요.
```

먼저 5.6의 질문을 `강아지에게 초콜릿을 먹여도 돼?`로 바꿔 답을 봐 둔 뒤, 그 질문을 둔 채 값을 붙여 넣고 이 셀부터 5.6까지 차례로 다시 실행합니다. 확인했으면 `답하세요.`로 되돌리고 이 셀부터 다시 실행합니다.

In [ ]:
llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # ✍️ CHANGE_HERE: 지시문 끝에 '문서에서 확인할 수 없습니다' 규칙을 더하면 약관에 없는 질문의 답은 어떻게 바뀔까요?
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)

<details><summary>무슨 일이 생기나</summary>

5.6의 질문이 "강아지에게 초콜릿을 먹여도 돼?"일 때입니다.

- 바꾸기 전: "약관에는 정보가 없다"고 하면서도 먼저 "아니요."(또는 "안 됩니다.")라고 답하는 일이 잦고(20번 중 14번), "먹이지 말고 동물병원에 문의하세요" 같은 조언도 자주 덧붙입니다(20번 중 17번). 약관에 없는 내용을 모델이 아는 상식으로 채운 것입니다.
- 바꾼 뒤: 매번 `문서에서 확인할 수 없습니다.`로 답하고, 거의 항상 그 한 문장으로 끝납니다(20번 중 18번. 나머지 2번은 "약관에 내용이 없다"는 설명이 한 줄 붙었습니다).
- 약관에 있는 질문("청약을 철회할 수 있는 기간은 며칠이야?")은 규칙을 더해도 "보험증권을 받은 날부터 15일 이내"라고 그대로 답합니다.

</details>

### 5.4 검색 결과를 context로 합치기

retriever가 돌려준 `Document` 목록을 프롬프트의 `{context}` 자리에 넣을 문자열 하나로 합칩니다. 청크 사이는 빈 줄로 구분합니다.

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### 5.5 RAG 체인 연결하기

retriever, 프롬프트, 모델, 출력 파서를 `|`로 이어 RAG 체인을 만듭니다. 체인에 질문 문자열 하나를 넣으면, 질문은 retriever와 `format_docs`를 거쳐 `context`가 되고, 원래 질문은 그대로 `question`이 되어 프롬프트에 들어가야 합니다.

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 5.6 질문하고 스트리밍으로 답변 받기

**✍️ CHANGE_HERE: 질문을 약관에 있는 내용과 없는 내용으로 바꾸면 답은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `펫보험 보장 내용이 뭐야?` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

약관에 있는 질문
```text
청약을 철회할 수 있는 기간은 며칠이야?
```

약관에 없는 질문
```text
강아지에게 초콜릿을 먹여도 돼?
```

이 셀만 다시 실행합니다(디버그 기록이 길면 맨 아래 `"output":` 뒤가 답입니다). 약관에 없는 질문의 답을 본 뒤에는 그 질문을 둔 채 5.3으로 가서 지시문을 바꿔 보고, 다 보면 둘 다 되돌립니다.

In [ ]:
print("RAG ready")
# ✍️ CHANGE_HERE: 질문을 약관에 있는 내용과 없는 내용으로 바꾸면 답은 어떻게 달라질까요?
question = "펫보험 보장 내용이 뭐야?"
for message in rag_chain.stream(question):
    print(message, end="")
print("\n")

<details><summary>무슨 일이 생기나</summary>

- 약관에 있는 질문: "보험증권을 받은 날부터 15일 이내에 청약을 철회할 수 있습니다."처럼 약관 내용으로 답합니다.
- 약관에 없는 질문: 4.3에서 본 관계없는 청크 3개가 context로 들어갑니다. 모델은 "약관에는 정보가 없다"고 하면서도 자주 "아니요"라고 답하거나 조언을 덧붙입니다. 가끔은 "정보가 없습니다"로만 끝납니다. 이 답이 5.3의 규칙으로 어떻게 바뀌는지 봅니다.

</details>

**볼 지점**
- 답변이 약관 내용에 기대고 있는지 봅니다. system 지시문이 "주어진 context만 근거로 답하세요"라고 요구합니다.
- 약관에 없는 질문에도 검색기는 청크 3개를 넘깁니다(4.3에서 본 것). 규칙을 더하기 전과 뒤에 모델이 그 청크를 받고 어떻게 답하는지 비교합니다.

## 6. RAGaaS

RAGaaS(RAG as a Service)는 청킹·임베딩·벡터 저장소 관리를 API가 대신 해 주는 방식입니다. 앞 장에서 직접 만든 흐름(나누기 → 임베딩 → 저장 → 검색)이 API 호출 두 번(문서 업로드, 검색)으로 줄어듭니다.

여럿이 같은 인덱스(검색용 문서 묶음)를 쓰므로 문서 업로드와 삭제는 강사가 따로 하고, 코드에는 주석으로 막아 두었습니다. 검색과 답변 만들기를 실행하고, `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 봅니다.

준비 코드의 출력이 `RAGaaS 모드: mock`이면 연습용 모의 응답입니다. 검색 결과는 늘 1건이고, 실제 약관 내용이 아닌 연습용 문장입니다. 그래서 2부는 `검색된 문서 수: 1`과 연습용 문장 하나를 출력하고, 3부 답변은 "주어진 자료에 답이 없다"는 취지로 나오는 것이 정상입니다.

**왜 배우나:** 문서를 올리고 검색하는 일을 RAGaaS API에 맡기면 직접 만들 코드가 줄어듭니다. 직접 만든 RAG와 단계가 어떻게 대응하는지 알아 두면 둘 중 무엇을 쓸지 고를 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고, RAGaaS에 보낼 인증 정보와 인덱스 이름을 준비합니다. 키는 화면에 출력하지 않습니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
RAG as a Service (RAGaaS): 삼성 사내 RAG API 활용

기존 방식 vs RAGaaS 비교:
  기존: 청킹 → 임베딩 클래스 구현 → 벡터스토어 생성 → 검색
  RAGaaS: 문서 업로드(API 1회) → 검색(API 1회)
  청킹·임베딩·벡터스토어 관리를 API가 대신 처리한다.

사외에서는 RAG_PROVIDER=mock으로 요청·응답 형식을 연습한다.
실제 문서 내용이 필요한 실습은 사내망에서 RAG_PROVIDER=samsung으로 실행한다.

"""

import os
from pathlib import Path

from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다

from config.llm import get_llm
from config.ragaas import post_ragaas

load_dotenv()

RAG_PROVIDER = os.getenv("RAG_PROVIDER", "samsung").strip().lower()
RAG_API_KEY = os.getenv("rag_api_key")
# 사용할 인덱스 네임
INDEX_NAME = "rp-aicoa-meritz-pet"
# 실제 사내 요청의 api-key 헤더에는 RAG_API_KEY가 들어간다.
print(f"RAGaaS 모드: {RAG_PROVIDER}")

### 6.1 1부: 업로드할 문서 준비

In [ ]:
# ── 1부: 문서 업로드 ──────────────────────────────────────────────────────────
# insert-doc 엔드포인트 하나로 청킹·임베딩·벡터스토어 저장을 모두 처리한다.
# chunk_factor: 청킹 방식과 크기를 서버에 위임 (직접 구현 불필요)

# 업로드할 텍스트 준비 (예: 파일을 읽어서 문자열로 전달)
PDF_PATH = Path("data/meritz_pet_insurance.pdf")

# pdfminer 로 텍스트 추출 (RAGaaS는 raw text를 받음)
from langchain_community.document_loaders import PDFMinerLoader

docs = PDFMinerLoader(str(PDF_PATH.resolve())).load()
raw_text = "\n".join(doc.page_content for doc in docs)

payload = {
    "index_name": INDEX_NAME,
    "data": {
        "doc_id": "meritz_pet_insurance",
        "title": "메리츠 펫보험 약관",
        "content": raw_text,
    },
    "chunk_factor": {
        "logic": "fixed_size",
        "chunk_size": 500,
        "chunk_overlap": 75,
        "separator": " ",
    },
}

print("=== 문서 업로드 ===")
# 문서 업로드(insert-doc)는 강사가 수업 전에 한 번 해 둡니다. 여럿이 같은 인덱스에 올리지 않도록 아래 줄은 실행하지 않습니다.
# response = post_ragaas("insert-doc", payload, api_key=RAG_API_KEY)
# print(f"상태: {response.status_code}")

### 6.2 2부: 문서 검색

**✍️ CHANGE_HERE: 질문을 4.3에서 쓴 질문으로 바꾸면 검색된 글과 답은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `펫보험 보장 내용이 뭐야?` 자리에 복사해 붙여 넣기

```text
청약을 철회할 수 있는 기간은 며칠이야?
```

2부와 3부를 다시 실행해 4.3·5.6의 결과와 비교합니다. 확인했으면 원래 질문으로 되돌립니다.

In [ ]:
# ── 2부: 문서 검색 ────────────────────────────────────────────────────────────
# retrieve-* 엔드포인트: bm25(키워드), knn(의미), rrf(하이브리드) 중 선택

# ✍️ CHANGE_HERE: 질문을 4.3에서 쓴 질문으로 바꾸면 검색된 글과 답은 어떻게 될까요?
question = "펫보험 보장 내용이 뭐야?"

search_payload = {
    "index_name": INDEX_NAME,
    "permission_groups": ["rag-public"],
    "query_text": question,
    "num_result_doc": 3,
    "fields_exclude": ["v_merge_title_content"],
}

print("=== 문서 검색 (RRF: 하이브리드) ===")

# retrieve-rrf 엔드포인트로 관련 문서를 검색합니다.

response = post_ragaas("retrieve-rrf", search_payload, api_key=RAG_API_KEY)
result = response.json()
hits = result["hits"]["hits"]
context_list = [hit["_source"]["merge_title_content"] for hit in hits]
context = "\n\n".join(context_list)
print(f"검색된 문서 수: {len(hits)}")
for index, content in enumerate(context_list):
    text = " ".join(content.split())
    print(f"{index}. len={len(content)} \n- {text[:180]}...")
    print()

<details><summary>무슨 일이 생기나</summary>

준비 코드 출력이 `RAGaaS 모드: mock`일 때입니다.

- 2부: `검색된 문서 수: 1`이고, `0.` 아래 글이 `[MOCK RAGaaS] 질문: 청약을 철회할 수 있는 기간은 며칠이야?. 이 문장은 … 모의 자료이며 …`로 바뀝니다. 모의 응답은 질문을 넣은 연습용 문장 하나만 돌려줍니다.
- 3부: "제공된 context에는 청약 철회 기간이 나와 있지 않습니다" 같은 답이 나옵니다. 같은 질문에 5.6은 "15일 이내"라고 답했습니다. 답은 검색된 글만큼만 좋아집니다.
- `RAGaaS 모드: samsung`이면(예상) 2부에 약관 인덱스에서 찾은 글이 최대 3개 나오고, 4.3의 결과와 비교할 수 있습니다.

</details>

### 6.3 3부: 검색 결과로 답변 만들기

In [ ]:
# ── 3부: RAG 체인 ─────────────────────────────────────────────────────────────
# 검색 결과를 context로 주입해 LLM이 문서 기반으로 답변하도록 한다.
# get_llm(): 로컬은 OpenAI, 사내망은 삼성 LLM으로 자동 전환


from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = get_llm()

# 모델에게 입력할 지시사항과, 사용자의 질문을 간단하게 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)

chain = prompt | llm | StrOutputParser()

print(f"=== 질문: {question} ===")
for chunk in chain.stream({"context": context, "question": question}):
    print(chunk, end="", flush=True)
print("\n")

### 6.4 4부: 문서 삭제(실행하지 않음)

In [ ]:
# ── 4부: 문서 삭제 (선택) ─────────────────────────────────────────────────────
# 실습 종료 후 업로드한 인덱스를 정리한다.

delete_payload = {
    "index_name": INDEX_NAME,
    "permission_groups": ["rag-public"],
    "doc_id": "meritz_pet_insurance",
}

# 문서 삭제(delete-doc)도 강사가 수업이 끝난 뒤 한 번만 합니다. 아래 줄은 실행하지 않습니다.
# response = post_ragaas("delete-doc", delete_payload, api_key=RAG_API_KEY)
# print(f"=== 문서 삭제: {response.status_code} ===")

**볼 지점**
- 1부의 업로드 요청(`payload`)에 `chunk_factor`가 있습니다. 청킹 방식과 크기를 서버에 맡기는 설정입니다. 크기(`chunk_size`, `chunk_overlap`)는 앞 장과 같은 값이지만, 나누는 방식(`logic`)은 `fixed_size`라 앞 장의 `RecursiveCharacterTextSplitter`와 다릅니다.
- 2부의 검색 요청(`search_payload`)에는 인덱스 이름, 질문(`query_text`), 가져올 문서 수(`num_result_doc`)가 들어갑니다. 임베딩 코드는 보이지 않습니다.
- 2부는 찾은 글의 앞 180자를 4.3과 같은 모양(`0. len=…`)으로 보여 줍니다.
- 검색 결과의 본문은 `hits` → `_source` → `merge_title_content` 경로로 꺼냅니다.
- 3부의 체인에는 retriever가 없습니다. 검색은 2부의 API가 이미 했습니다.

## 7. 정리

약관 PDF 한 권으로 RAG의 여섯 단계를 차례로 만들었습니다.

| 장 | 단계 | 이 노트북에서 한 일 |
|---|---|---|
| 2. PDF 불러오기와 청크 나누기 | Loading, Splitting | `PDFMinerLoader`로 읽고 `RecursiveCharacterTextSplitter`로 나눔 |
| 3. 임베딩과 유사도 | Embedding | 문장을 벡터로 바꾸고 코사인 유사도로 비교 |
| 4. 벡터 저장소와 검색 | Storage, Retrieval | Chroma에 넣고 retriever로 가까운 청크 3개 찾기 |
| 5. RAG 체인 | Generation | 검색 결과를 `context`로 넣어 답변을 스트리밍으로 받기 |
| 6. RAGaaS | 전 단계 | 업로드·검색을 API에 맡기고 답변만 만들기 |

청크 크기, 가져올 청크 수(`k`), 질문, system 지시문을 바꿔 다시 실행하며 결과를 비교했습니다. 낱말이 달라도, 언어가 달라도 뜻이 가까우면 검색된다는 것과, 검색기는 약관에 없는 질문에도 청크를 돌려주므로 "문서에서 확인할 수 없습니다" 같은 프롬프트 규칙이 필요하다는 것을 확인했습니다.